# Text Classification with Naive Bayes & TF-IDF
Accompanies **Unit 4.2: Naive Bayes Classification**.

Naive Bayes applies Bayes' Theorem with the conditional independence assumption:
$$P(y = c \mid \mathbf{x}) \propto P(y = c) \prod_{j=1}^p P(x_j \mid y = c)$$

For text documents, **Multinomial Naive Bayes** treats each word occurrence as a multinomial draw. When a word in testing was never observed during training, $P(w_j \mid y=c) = 0$, causing the entire probability product to collapse to zero (the zero-frequency problem). 
We resolve this via **Laplace Smoothing**:
$$\hat{\theta}_{jc} = \frac{N_{jc} + \alpha}{N_c + \alpha V}$$
where $\alpha > 0$ is the pseudo-count smoothing hyperparameter.

In this notebook, we classify financial consumer complaints from `Consumer_Complaints_train.csv.zip` and tune smoothing parameter $\alpha$.

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

zip_path = os.path.abspath('../../data/Consumer_Complaints_train.csv.zip')
df = pd.read_csv(zip_path, compression='zip')

narrative_col = 'Consumer complaint narrative'
target_col = 'Product'

clean_df = df[[narrative_col, target_col]].dropna().sample(n=6000, random_state=42)
print("Top product categories:")
print(clean_df[target_col].value_counts().head(4))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(clean_df[narrative_col], clean_df[target_col], test_size=0.25, random_state=42)

### TF-IDF Vectorization and Hyperparameter Tuning for Smoothing $\alpha$
- Small $\alpha$ (e.g. $0.01$): Weak shrinkage, trusts observed training frequencies closely.
- Moderate $\alpha$ ($\alpha = 1.0$, Laplace): Standard uninformative uniform Dirichlet prior.
- Large $\alpha$ (e.g. $10.0$): Over-smooths word likelihoods toward uniform distribution, increasing bias.

In [ ]:
vectorizer = TfidfVectorizer(max_features=4000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

# Tuning alpha over log-spaced grid
param_grid = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]}
grid = GridSearchCV(MultinomialNB(), param_grid=param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_vec, y_train)

print(f"Optimal Laplace alpha: {grid.best_params_['alpha']} | Best CV Accuracy: {grid.best_score_:.4f}")

best_nb = grid.best_estimator_
y_pred = best_nb.predict(X_test_vec)
print(classification_report(y_test, y_pred))